# PlotPot mass photometry notebook (Colab version)

Fits multi-Gaussian models to mass photometry landing-event distributions and produces publication-quality plots.

**Input format:** Refeyn DiscoverMP `.csv` export (columns: `contrasts`, `masses_kDa`, `selections`, …) or `.h5`.

**Workflow** — run cells top to bottom:
1. **Dependencies · Imports** — run once.
2. **Upload** — select your `.csv` export; check whether masses are already calibrated.
3. **Calibration** *(optional)* — skip if `masses_kDa` is already populated; otherwise upload a calibration file and enter known masses.
4. **Parameters** — set histogram range, bin width, number of Gaussians, and initial peak positions.
5. **Fit & plot** — fits multi-Gaussian model, prints results table, shows publication plot.
6. **Save & download** — PDF + PNG.

In [1]:
#@title Step 1 · Install dependencies & import { display-mode: "form" }
import importlib, subprocess, sys

_required = ['numpy', 'pandas', 'matplotlib', 'scipy', 'h5py']
_missing  = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)
    print(f'Installed: {_missing}')

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from scipy.optimize import curve_fit
from scipy.stats import linregress

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})
print('Ready.')

Ready.


In [2]:
#@title Step 2 · Upload data { display-mode: "form" }
from google.colab import files as _colab_files

print('Select your mass photometry export (.csv or .h5):')
_uploaded = _colab_files.upload()
_fname    = next(iter(_uploaded))
DATA_FILE = Path(_fname)


def _load_csv(text):
    df = pd.read_csv(io.StringIO(text))
    df.columns = [c.strip() for c in df.columns]
    return df


def _load_h5(data):
    import h5py
    with h5py.File(io.BytesIO(data), 'r') as f:
        def _get(keys):
            for k in keys:
                try:
                    return f[k][:]
                except Exception:
                    pass
            return None
        contrasts  = _get(['/events/contrasts',  'contrasts'])
        masses     = _get(['/events/masses_kDa', 'masses_kDa'])
        selections = _get(['/events/selections',  'selections'])
    d = {'contrasts': contrasts}
    if masses    is not None: d['masses_kDa'] = masses
    if selections is not None: d['selections'] = selections
    return pd.DataFrame(d)


_ext = DATA_FILE.suffix.lower()
if _ext == '.csv':
    df_raw = _load_csv(_uploaded[_fname].decode('utf-8'))
elif _ext in ('.h5', '.hdf5'):
    df_raw = _load_h5(_uploaded[_fname])
else:
    raise ValueError(f'Unsupported format: {_ext}  (use .csv or .h5)')

# Apply selections filter if present
if 'selections' in df_raw.columns:
    df_events = df_raw[df_raw['selections'] == 1.0].copy().reset_index(drop=True)
else:
    df_events = df_raw.copy()

# Detect whether mass calibration has already been applied
_has_masses = ('masses_kDa' in df_events.columns
               and df_events['masses_kDa'].notna().sum() > 10)

print(f'Loaded {_fname!r}')
print(f'Total events : {len(df_raw):,}   selected : {len(df_events):,}')
print(f'Contrasts    : {df_events["contrasts"].min():.4f} – {df_events["contrasts"].max():.4f}')
if _has_masses:
    _m = df_events['masses_kDa'].dropna()
    print(f'Masses       : {_m.min():.0f} – {_m.max():.0f} kDa  (median {_m.median():.0f} kDa)')
    print()
    print('Masses already calibrated — you can skip Step 3.')
else:
    print()
    print('No mass data found — run Step 3 to calibrate.')

ModuleNotFoundError: No module named 'google'

In [ ]:
#@title Step 3 · Calibration (skip if masses_kDa is already populated) { display-mode: "form" }
#@markdown Upload a calibration CSV (same format as sample), then enter the known masses
#@markdown that correspond to the peaks detected in the calibration contrast histogram.

#@markdown **Known masses (kDa)** — comma-separated, matching peaks low → high contrast magnitude.
KNOWN_MASSES_KDA = "66, 480" #@param {type:"string"}
#@markdown **Calibration contrast bin width** — for histogram used to find peaks.
CAL_BIN_WIDTH = 0.002 #@param {type:"number"}

if _has_masses:
    print('Masses already present — calibration skipped.')
else:
    from google.colab import files as _colab_files2
    from scipy.signal import find_peaks as _fp

    print('Select calibration file:')
    _cal_up   = _colab_files2.upload()
    _cal_fname = next(iter(_cal_up))
    df_cal    = _load_csv(_cal_up[_cal_fname].decode('utf-8'))
    if 'selections' in df_cal.columns:
        df_cal = df_cal[df_cal['selections'] == 1.0]

    _known = [float(x.strip()) for x in KNOWN_MASSES_KDA.split(',') if x.strip()]

    # Contrast histogram of calibration data
    _c = df_cal['contrasts'].dropna().values
    _bins_cal = np.arange(_c.min(), _c.max() + CAL_BIN_WIDTH, CAL_BIN_WIDTH)
    _counts_cal, _edges_cal = np.histogram(_c, bins=_bins_cal)
    _centers_cal = 0.5 * (_edges_cal[:-1] + _edges_cal[1:])

    # Auto-detect peaks (sorted by contrast magnitude, most negative = highest mass)
    _pk_idx, _ = _fp(_counts_cal, height=_counts_cal.max() * 0.1,
                      distance=int(0.02 / CAL_BIN_WIDTH), prominence=_counts_cal.max() * 0.05)
    _peak_contrasts = _centers_cal[_pk_idx]
    # Sort by contrast (most negative → highest mass if slope < 0)
    _peak_contrasts = np.sort(_peak_contrasts)

    if len(_peak_contrasts) < len(_known):
        print(f'WARNING: {len(_peak_contrasts)} peaks found, but {len(_known)} known masses provided.')
        print(f'Detected peak contrasts: {_peak_contrasts}')
        print('Check CAL_BIN_WIDTH or adjust KNOWN_MASSES_KDA.')
    else:
        _peak_contrasts_used = _peak_contrasts[:len(_known)]
        _slope, _intercept, _r, _, _ = linregress(_peak_contrasts_used, _known)
        _r2 = _r ** 2

        print(f'Calibration peaks detected: {_peak_contrasts_used}')
        print(f'Known masses (kDa)        : {_known}')
        print(f'Linear fit  :  mass = {_slope:.1f} × contrast + {_intercept:.1f} kDa')
        print(f'R²          :  {_r2:.4f}')

        # Apply calibration to sample
        df_events['masses_kDa'] = _slope * df_events['contrasts'] + _intercept
        _has_masses = True
        _m = df_events['masses_kDa']
        print(f'\nCalibrated masses: {_m.min():.0f} – {_m.max():.0f} kDa  (median {_m.median():.0f} kDa)')

In [ ]:
#@title Step 4 · Parameters { display-mode: "form" }
#@markdown **Histogram range and binning**
MASS_MIN  = 50.0   #@param {type:"number"}
MASS_MAX  = 800.0  #@param {type:"number"}
BIN_WIDTH = 10.0   #@param {type:"number"}

#@markdown ---
#@markdown **Gaussian peaks** — number of peaks and initial guesses (kDa, comma-separated).
N_GAUSSIANS   = 1    #@param {type:"integer"}
PEAK_GUESSES  = "150" #@param {type:"string"}
#@markdown **Tolerance** — how far the fitted mean may move from each guess (kDa); 0 = auto.
MEAN_TOLERANCE = 0.0 #@param {type:"number"}
#@markdown **Fit a flat baseline**
FIT_BASELINE   = False #@param {type:"boolean"}

#@markdown ---
#@markdown **Normalise counts** (fraction of total events)
NORMALIZE = False #@param {type:"boolean"}

# ── Auto peak detection to suggest guesses ─────────────────────────────────────
if _has_masses:
    from scipy.signal import find_peaks as _fp2
    _m_hist = df_events['masses_kDa'].dropna()
    _m_hist = _m_hist[(_m_hist >= MASS_MIN) & (_m_hist <= MASS_MAX)]
    _bins_p  = np.arange(MASS_MIN, MASS_MAX + BIN_WIDTH, BIN_WIDTH)
    _cts_p, _edg_p = np.histogram(_m_hist, bins=_bins_p)
    _cen_p  = 0.5 * (_edg_p[:-1] + _edg_p[1:])
    _pk2, _ = _fp2(_cts_p, height=_cts_p.max() * 0.1,
                    distance=int(50 / BIN_WIDTH), prominence=_cts_p.max() * 0.05)
    _suggested = ', '.join(f'{_cen_p[i]:.0f}' for i in _pk2)
    print(f'Events in range: {len(_m_hist):,}')
    print(f'Auto-detected peaks: {_suggested or "none"}')
    print(f'\nSet N_GAUSSIANS={len(_pk2)} and PEAK_GUESSES="{_suggested}" — or use your own.')
else:
    print('No masses available. Run Step 3 first.')

In [ ]:
#@title Step 5 · Fit & publication plot { display-mode: "form" }
#@markdown *Re-run after adjusting parameters in Step 4.*

assert _has_masses, 'No mass data — run Steps 3 and 4 first.'

COLORS = ['#1a4f8a', '#c0392b', '#27ae60', '#8e44ad', '#d35400']

# ── Build histogram ────────────────────────────────────────────────────────────
masses = df_events['masses_kDa'].dropna()
masses = masses[(masses >= MASS_MIN) & (masses <= MASS_MAX)].values
bins   = np.arange(MASS_MIN, MASS_MAX + BIN_WIDTH, BIN_WIDTH)
counts, edges = np.histogram(masses, bins=bins)
centers = 0.5 * (edges[:-1] + edges[1:])
y_data  = counts / counts.sum() if NORMALIZE else counts.astype(float)
y_label = 'Fraction' if NORMALIZE else 'Counts'

# ── Parse guesses & build model ────────────────────────────────────────────────
guesses = [float(x.strip()) for x in PEAK_GUESSES.split(',') if x.strip()]
assert len(guesses) == N_GAUSSIANS,     f'N_GAUSSIANS={N_GAUSSIANS} but {len(guesses)} guesses provided.'

_tol = MEAN_TOLERANCE if MEAN_TOLERANCE > 0 else (MASS_MAX - MASS_MIN) / 3


def _model(x, *params):
    off = 1 if FIT_BASELINE else 0
    y = np.full_like(x, params[0] if FIT_BASELINE else 0.0, dtype=float)
    for i in range(N_GAUSSIANS):
        a, mu, sig = params[off + 3*i], params[off + 3*i+1], params[off + 3*i+2]
        y += a * np.exp(-0.5 * ((x - mu) / sig) ** 2)
    return y


_y_max  = y_data.max()
_sig0   = max(BIN_WIDTH * 4, 30.0)
_amp0   = [float(y_data[np.argmin(np.abs(centers - g))]) for g in guesses]
_p0 = ([0.0] if FIT_BASELINE else []) + [v for g, a in zip(guesses, _amp0)
                                           for v in (a, g, _sig0)]
_lo = ([0.0] if FIT_BASELINE else []) + [v for g in guesses
                                           for v in (0.0, g - _tol, BIN_WIDTH)]
_hi = ([_y_max * 0.3] if FIT_BASELINE else []) + [v for g in guesses
                                                     for v in (_y_max * 3, g + _tol,
                                                                (MASS_MAX - MASS_MIN) / 2)]

popt, pcov = curve_fit(_model, centers, y_data, p0=_p0, bounds=(_lo, _hi), maxfev=20000)
perr = np.sqrt(np.diag(np.abs(pcov)))

off  = 1 if FIT_BASELINE else 0
_baseline_fit = popt[0] if FIT_BASELINE else 0.0

peaks = []
for i in range(N_GAUSSIANS):
    a      = popt[off + 3*i]
    mu     = popt[off + 3*i + 1]
    sig    = abs(popt[off + 3*i + 2])
    mu_err = perr[off + 3*i + 1]
    area   = a * sig * np.sqrt(2 * np.pi)
    peaks.append(dict(a=a, mu=mu, sig=sig, mu_err=mu_err, area=area))

total_area = sum(p['area'] for p in peaks)
for p in peaks:
    p['pct'] = 100 * p['area'] / total_area if total_area > 0 else np.nan

# ── Print results ──────────────────────────────────────────────────────────────
print('=== Gaussian fit results ===\n')
print(f'  {"#":>2}  {"Mass (kDa)":>14}  {"Sigma (kDa)":>12}  {"Events (%)":>10}')
for i, p in enumerate(peaks):
    print(f'  {i+1:>2}  {p["mu"]:>8.0f} ± {p["mu_err"]:>4.0f}  '
          f'  {p["sig"]:>8.0f}      '
          f'  {p["pct"]:>9.1f}')
print(f'\n  Events in range: {len(masses):,} / {len(df_events):,} total')

# ── Publication plot ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5.5, 4.0), layout='constrained')

# Histogram bars
ax.bar(centers, y_data, width=BIN_WIDTH * 0.9,
       color='#d0d0d0', edgecolor='#999999', linewidth=0.4, zorder=1)

# Gaussian curves
x_fine = np.linspace(MASS_MIN, MASS_MAX, 2000)
gauss_curves = []
for i, p in enumerate(peaks):
    color = COLORS[i % len(COLORS)]
    y_g   = p['a'] * np.exp(-0.5 * ((x_fine - p['mu']) / p['sig']) ** 2)
    gauss_curves.append(y_g)
    ax.plot(x_fine, y_g + _baseline_fit, color=color, lw=1.8, zorder=3)

    # Label — place on the emptier side of the peak
    x_rel = (p['mu'] - MASS_MIN) / (MASS_MAX - MASS_MIN) if MASS_MAX > MASS_MIN else 0.5
    x_ax, ha = (0.97, 'right') if x_rel < 0.5 else (0.03, 'left')
    from matplotlib.transforms import blended_transform_factory as _btf
    ax.text(x_ax, p['a'] * 0.55 + _baseline_fit,
            f"{p['mu']:.0f} kDa\n({p['pct']:.0f}%)",
            transform=_btf(ax.transAxes, ax.transData),
            color=color, fontsize=9, va='center', ha=ha,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.85))

# Sum of Gaussians (only when more than one)
if N_GAUSSIANS > 1:
    ax.plot(x_fine, sum(gauss_curves) + _baseline_fit,
            color='black', lw=1.5, ls='--', zorder=4, label='Sum')
    ax.legend(loc='upper right', frameon=True,
              facecolor='white', edgecolor='none', framealpha=0.85, fontsize=9)

ax.set_xlabel('Mass (kDa)', fontsize=11)
ax.set_ylabel(y_label, fontsize=11)
ax.set_xlim(MASS_MIN, MASS_MAX)
ax.set_ylim(bottom=0)
ax.set_title(DATA_FILE.stem, fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

In [ ]:
#@title Step 6 · Save & download { display-mode: "form" }
from google.colab import files as _colab_files

OUTPUT_STEM = DATA_FILE.stem
_outputs = []
for ext in ('pdf', 'png'):
    out = f'/content/{OUTPUT_STEM}_mass_photometry.{ext}'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')
    _outputs.append(out)

print('\nStarting downloads...')
for out in _outputs:
    _colab_files.download(out)